# Recalls to Neo4j Desktop (Local)

Este notebook carga Recalls a tu instancia local de Neo4j Desktop.

## ⚠️ Requisitos:

1. **Neo4j Desktop** instalado y ejecutándose
2. **Instancia creada** en el puerto 7687
3. **Archivo CSV** ya generado: `data/neo4j/exports/recalls_neo4j_ready.csv`

---

## ✅ Ventajas:

- 🚀 **Más rápido** que AuraDB
- 💾 **Sin límites** de capacidad
- 🎛️ **Control total** sobre datos
- 💰 **Sin costos** de cloud


In [1]:
import pandas as pd
from neo4j import GraphDatabase
from pathlib import Path

print("="*70)
print("CONFIGURACION NEO4J DESKTOP LOCAL")
print("="*70)

# Credenciales de Neo4j Desktop (LOCAL)
NEO4J_URI  = "bolt://localhost:7687"  # Local
NEO4J_USER = "neo4j"
NEO4J_PASS = "proyectotec"  # Cambiar a tu password real

# Inicializar driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# Verificar conexión
try:
    driver.verify_connectivity()
    print(f"[OK] Conectado a Neo4j Desktop en {NEO4J_URI}")
except Exception as e:
    print(f"[ERROR] No se puede conectar a Neo4j: {e}")
    print("[!] Asegurate de que Neo4j Desktop esté corriendo")


CONFIGURACION NEO4J DESKTOP LOCAL
[OK] Conectado a Neo4j Desktop en bolt://localhost:7687


## Crear Constraints e Índices


In [2]:
CONSTRAINTS = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (r:Recall)        REQUIRE r.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (i:Investigation) REQUIRE i.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Complaint)     REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Component)     REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (m:Make)          REQUIRE m.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (m:Model)         REQUIRE (m.name, m.make) IS UNIQUE",
    "CREATE INDEX IF NOT EXISTS FOR (c:Component) ON (c.name_lower)",
]

with driver.session(database="neo4j") as s:
    for q in CONSTRAINTS:
        try:
            s.run(q)
            print(f"[OK] {q[:60]}")
        except Exception as e:
            print(f"[SKIP] {q[:60]} - Ya existe")
print("Constraints listos ✅")


[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (r:Recall)        REQUIR
[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (i:Investigation) REQUIR
[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (c:Complaint)     REQUIR
[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (c:Component)     REQUIR
[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (m:Make)          REQUIR
[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (m:Model)         REQUIR
[OK] CREATE INDEX IF NOT EXISTS FOR (c:Component) ON (c.name_lowe
Constraints listos ✅


## Cypher para Subir Recalls


In [3]:
CYPHER_UPSERT_RECALL_HIER_5 = """
UNWIND $rows AS row
MERGE (r:Recall {id: row.campaign_no})
  SET r.camp_no          = row.campaign_no,
      r.recall_date      = CASE WHEN row.recall_date = '' THEN NULL ELSE row.recall_date END,
      r.make             = row.make_norm,
      r.model            = row.model_norm,
      r.year             = CASE WHEN row.year IS NULL OR row.year = '' THEN NULL ELSE toInteger(row.year) END,
      r.component        = row.component,
      r.subject          = coalesce(row.subject, ''),
      r.consequence      = coalesce(row.consequence, ''),
      r.corrective_action = coalesce(row.corrective_action, '')

FOREACH (_ IN CASE WHEN row.make_norm <> '' AND row.make_norm IS NOT NULL THEN [1] ELSE [] END |
  MERGE (mk:Make {name: row.make_norm})
  MERGE (r)-[:OF_MAKE]->(mk)
)

FOREACH (_ IN CASE WHEN row.model_norm <> '' AND row.model_norm IS NOT NULL AND row.make_norm <> '' AND row.make_norm IS NOT NULL THEN [1] ELSE [] END |
  MERGE (md:Model {name: row.model_norm, make: row.make_norm})
  MERGE (r)-[:OF_MODEL]->(md)
)

MERGE (c1:Component {name: row.comp_l1})
  ON CREATE SET c1.name_lower = toLower(row.comp_l1)
  ON MATCH  SET c1.name_lower = coalesce(c1.name_lower, toLower(row.comp_l1))

FOREACH (_ IN CASE WHEN row.comp_l2 <> '' THEN [1] ELSE [] END |
  MERGE (p1:Component {name: row.comp_l1})
  MERGE (c2:Component {name: row.comp_l2})
    ON CREATE SET c2.name_lower = toLower(row.comp_l2)
    ON MATCH  SET c2.name_lower = coalesce(c2.name_lower, toLower(row.comp_l2))
  MERGE (c2)-[:SUB_OF]->(p1)
)

FOREACH (_ IN CASE WHEN row.comp_l3 <> '' AND row.comp_l2 <> '' THEN [1] ELSE [] END |
  MERGE (p2:Component {name: row.comp_l2})
  MERGE (c3:Component {name: row.comp_l3})
    ON CREATE SET c3.name_lower = toLower(row.comp_l3)
    ON MATCH  SET c3.name_lower = coalesce(c3.name_lower, toLower(row.comp_l3))
  MERGE (c3)-[:SUB_OF]->(p2)
)

FOREACH (_ IN CASE WHEN row.comp_l4 <> '' AND row.comp_l3 <> '' THEN [1] ELSE [] END |
  MERGE (p3:Component {name: row.comp_l3})
  MERGE (c4:Component {name: row.comp_l4})
    ON CREATE SET c4.name_lower = toLower(row.comp_l4)
    ON MATCH  SET c4.name_lower = coalesce(c4.name_lower, toLower(row.comp_l4))
  MERGE (c4)-[:SUB_OF]->(p3)
)

FOREACH (_ IN CASE WHEN row.comp_l5 <> '' AND row.comp_l4 <> '' THEN [1] ELSE [] END |
  MERGE (p4:Component {name: row.comp_l4})
  MERGE (c5:Component {name: row.comp_l5})
    ON CREATE SET c5.name_lower = toLower(row.comp_l5)
    ON MATCH  SET c5.name_lower = coalesce(c5.name_lower, toLower(row.comp_l5))
  MERGE (c5)-[:SUB_OF]->(p4)
)

WITH r,
     CASE
       WHEN row.comp_l5 <> '' THEN row.comp_l5
       WHEN row.comp_l4 <> '' THEN row.comp_l4
       WHEN row.comp_l3 <> '' THEN row.comp_l3
       WHEN row.comp_l2 <> '' THEN row.comp_l2
       ELSE row.comp_l1
     END AS target_comp
MATCH (cx:Component {name: target_comp})
MERGE (r)-[:MENTIONS]->(cx)

RETURN count(r) AS upserted;
"""


In [4]:
CSV = Path("../data/neo4j/exports/recalls_neo4j_ready.csv")
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
print(f"[i] CSV cargado: {len(df):,} filas")
print(f"[i] Columnas: {list(df.columns)}")


[i] CSV cargado: 12,760 filas
[i] Columnas: ['campaign_no', 'make_norm', 'model_norm', 'year', 'component', 'recall_date', 'subject', 'consequence', 'corrective_action', 'comp_l1', 'comp_l2', 'comp_l3']


In [5]:
def ingest(csv_df, cypher, batch=400):
    total, i = len(csv_df), 0
    with driver.session(database="neo4j") as s:
        while i < total:
            rows = csv_df.iloc[i:i+batch].to_dict('records')
            s.run(cypher, rows=rows)
            i += batch
            print(f"→ {min(i,total)}/{total}")
    print("Ingesta completa ✅")

# Test con 200 filas
print("[i] Probando con 200 filas...")
ingest(df.head(200), CYPHER_UPSERT_RECALL_HIER_5, batch=200)

# Verificación
with driver.session(database="neo4j") as s:
    recalls = s.run("MATCH (r:Recall) RETURN count(r) AS n").single()['n']
    comps = s.run("MATCH (c:Component) RETURN count(c) AS n").single()['n']
    print(f"\n[OK] Test exitoso. Recalls: {recalls:,}, Components: {comps}")

# Ingesta total
print("\n" + "="*70)
print("INGESTA COMPLETA DE RECALLS")
print("="*70)
ingest(df, CYPHER_UPSERT_RECALL_HIER_5, batch=400)

# Verificación final
with driver.session(database="neo4j") as s:
    recalls = s.run("MATCH (r:Recall) RETURN count(r) AS n").single()['n']
    comps = s.run("MATCH (c:Component) RETURN count(c) AS n").single()['n']
    makes = s.run("MATCH (m:Make) RETURN count(m) AS n").single()['n']
    
    print(f"\n[OK] Ingesta completada!")
    print(f"   Recalls: {recalls:,}")
    print(f"   Components: {comps}")
    print(f"   Makes: {makes}")


[i] Probando con 200 filas...
→ 200/200
Ingesta completa ✅

[OK] Test exitoso. Recalls: 200, Components: 106

INGESTA COMPLETA DE RECALLS
→ 400/12760
→ 800/12760
→ 1200/12760
→ 1600/12760
→ 2000/12760
→ 2400/12760
→ 2800/12760
→ 3200/12760
→ 3600/12760
→ 4000/12760
→ 4400/12760
→ 4800/12760
→ 5200/12760
→ 5600/12760
→ 6000/12760
→ 6400/12760
→ 6800/12760
→ 7200/12760
→ 7600/12760
→ 8000/12760
→ 8400/12760
→ 8800/12760
→ 9200/12760
→ 9600/12760
→ 10000/12760
→ 10400/12760
→ 10800/12760
→ 11200/12760
→ 11600/12760
→ 12000/12760
→ 12400/12760
→ 12760/12760
Ingesta completa ✅

[OK] Ingesta completada!
   Recalls: 12,760
   Components: 500
   Makes: 888
